# Phase 1 — Real Data Exploration
zanbil.ir Nginx access log — 200k-line HDFS sample.
Spark runs in `local[4]` mode inside the Jupyter container (avoids driver/executor JAR version mismatch with external cluster).

In [1]:
import os, sys
os.environ['SPARK_HOME'] = '/usr/local/spark-3.5.0-bin-hadoop3'
sys.path.insert(0, '/usr/local/spark-3.5.0-bin-hadoop3/python')
sys.path.insert(0, '/usr/local/spark-3.5.0-bin-hadoop3/python/lib/py4j-0.10.9.7-src.zip')
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import LongType

spark = (
    SparkSession.builder
    .appName('01_exploration')
    .master('local[4]')
    .config('spark.hadoop.fs.defaultFS', 'hdfs://hdfs-namenode:9000')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', '2g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)
print('Mode:', spark.sparkContext.master)


Spark version: 3.5.0
Mode: local[4]


In [2]:
raw = spark.read.text('hdfs://hdfs-namenode:9000/raw/access-logs/access.log')
total_lines = raw.count()
print(f'Total lines: {total_lines:,}')
print('--- First 3 real lines ---')
for r in raw.limit(3).collect():
    print(repr(r.value))


Total lines: 200,000
--- First 3 real lines ---
'54.36.149.41 - - [22/Jan/2019:03:56:14 +0330] "GET /filter/27|13%20%D9%85%DA%AF%D8%A7%D9%BE%DB%8C%DA%A9%D8%B3%D9%84,27|%DA%A9%D9%85%D8%AA%D8%B1%20%D8%A7%D8%B2%205%20%D9%85%DA%AF%D8%A7%D9%BE%DB%8C%DA%A9%D8%B3%D9%84,p53 HTTP/1.1" 200 30577 "-" "Mozilla/5.0 (compatible; AhrefsBot/6.1; +http://ahrefs.com/robot/)" "-"'
'31.56.96.51 - - [22/Jan/2019:03:56:16 +0330] "GET /image/60844/productModel/200x200 HTTP/1.1" 200 5667 "https://www.zanbil.ir/m/filter/b113" "Mozilla/5.0 (Linux; Android 6.0; ALE-L21 Build/HuaweiALE-L21) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/66.0.3359.158 Mobile Safari/537.36" "-"'
'31.56.96.51 - - [22/Jan/2019:03:56:16 +0330] "GET /image/61474/productModel/200x200 HTTP/1.1" 200 5379 "https://www.zanbil.ir/m/filter/b113" "Mozilla/5.0 (Linux; Android 6.0; ALE-L21 Build/HuaweiALE-L21) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/66.0.3359.158 Mobile Safari/537.36" "-"'


In [3]:
REGEX = (
    r'^(\S+) \S+ \S+ \[([^\]]+)\] '
    r'"(\S+) (\S+) ([^"]+)" (\d{3}) (\S+) '
    r'"([^"]*)" "([^"]*)"(?: "([^"]*)")?'
)
parsed = raw.select(
    F.col('value').alias('_raw'),
    F.regexp_extract('value', REGEX, 1).alias('ip'),
    F.regexp_extract('value', REGEX, 2).alias('ts_raw'),
    F.regexp_extract('value', REGEX, 3).alias('method'),
    F.regexp_extract('value', REGEX, 4).alias('path'),
    F.regexp_extract('value', REGEX, 5).alias('protocol'),
    F.regexp_extract('value', REGEX, 6).alias('status_s'),
    F.regexp_extract('value', REGEX, 7).alias('bytes_s'),
    F.regexp_extract('value', REGEX, 8).alias('referrer'),
    F.regexp_extract('value', REGEX, 9).alias('user_agent'),
    F.regexp_extract('value', REGEX, 10).alias('xff'),
).withColumn('_parse_ok', F.col('ip') != '')

n_ok   = parsed.filter(F.col('_parse_ok')).count()
n_fail = parsed.filter(~F.col('_parse_ok')).count()
print(f'Parsed OK : {n_ok:,}  ({100*n_ok/total_lines:.4f}%)')
print(f'Failed    : {n_fail:,}  ({100*n_fail/total_lines:.4f}%)')
print('--- Sample FAILED lines ---')
parsed.filter(~F.col('_parse_ok')).select('_raw').show(5, truncate=False)


Parsed OK : 199,998  (99.9990%)
Failed    : 2  (0.0010%)
--- Sample FAILED lines ---
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|_raw                                                                                                                                                                                                                                                                                                                                                               |
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [4]:
print('--- HTTP Status code distribution ---')
parsed.filter(F.col('_parse_ok')).groupBy('status_s').count().orderBy('status_s').show(30)


--- HTTP Status code distribution ---
+--------+------+
|status_s| count|
+--------+------+
|     200|186280|
|     301|  1960|
|     302|  5196|
|     304|  3433|
|     400|     8|
|     403|   152|
|     404|  2274|
|     408|    10|
|     499|   672|
|     500|     5|
|     502|     7|
|     504|     1|
+--------+------+



In [5]:
bytes_df = parsed.filter(F.col('_parse_ok')).select(
    F.when(F.col('bytes_s') == '-', None)
     .otherwise(F.col('bytes_s').cast(LongType())).alias('bytes')
)
n_null = bytes_df.filter(F.col('bytes').isNull()).count()
n_tot  = bytes_df.count()
print(f'bytes == "-" (null): {n_null:,}  ({100*n_null/n_tot:.2f}%)')
pcts = bytes_df.filter(F.col('bytes').isNotNull()).approxQuantile(
    'bytes', [0.5, 0.95, 0.99, 0.999, 1.0], 0.001
)
for l, v in zip(['p50','p95','p99','p99.9','max'], pcts):
    print(f'  {l}: {int(v):,} bytes  ({int(v)/1024:.1f} KB)')


bytes == "-" (null): 0  (0.00%)
  p50: 4,167 bytes  (4.1 KB)
  p95: 42,673 bytes  (41.7 KB)
  p99: 93,044 bytes  (90.9 KB)
  p99.9: 1,249,490 bytes  (1220.2 KB)
  max: 1,249,490 bytes  (1220.2 KB)


In [6]:
dim = (
    spark.read
    .option('header', 'true')
    .option('inferSchema', 'false')
    .csv('hdfs://hdfs-namenode:9000/raw/client-hostname/client_hostname.csv')
)
print('Columns:', dim.columns)
print('Total rows:', dim.count())
res = dim.filter(F.col('hostname') != F.col('client')).count()
unr = dim.filter(F.col('hostname') == F.col('client')).count()
print(f'Resolved hostname: {res:,}  Unresolved (IP==hostname): {unr:,}')
dim.filter(F.col('hostname') == F.col('client')).show(3, truncate=80)


Columns: ['client', 'hostname', 'alias_list', 'address_list']
Total rows: 258445
Resolved hostname: 5,815  Unresolved (IP==hostname): 252,626
+------------+------------+----------------------+------------+
|      client|    hostname|            alias_list|address_list|
+------------+------------+----------------------+------------+
|5.123.144.95|5.123.144.95|[Errno 1] Unknown host|        NULL|
|5.122.76.187|5.122.76.187|[Errno 1] Unknown host|        NULL|
|5.215.249.99|5.215.249.99|[Errno 1] Unknown host|        NULL|
+------------+------------+----------------------+------------+
only showing top 3 rows



In [7]:
log_ips = parsed.filter(F.col('_parse_ok')).select('ip').distinct()
dim_ips = dim.select(F.col('client').alias('ip')).distinct()
n_log   = log_ips.count()
n_match = log_ips.join(dim_ips, 'ip', 'inner').count()
print(f'Distinct IPs in log : {n_log:,}')
print(f'Matched in dim      : {n_match:,}  ({100*n_match/n_log:.1f}%)')
print(f'Not in dim          : {n_log - n_match:,}')


Distinct IPs in log : 6,510
Matched in dim      : 6,507  (100.0%)
Not in dim          : 3


In [8]:
print('--- Top 30 user agents ---')
(
    parsed.filter(F.col('_parse_ok'))
    .groupBy('user_agent').count()
    .orderBy(F.desc('count'))
    .show(30, truncate=90)
)


--- Top 30 user agents ---
+------------------------------------------------------------------------------------------+-----+
|                                                                                user_agent|count|
+------------------------------------------------------------------------------------------+-----+
|Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/71.0.3578.98...|16192|
|Mozilla/5.0 (Windows NT 6.1; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/...|14137|
|                   Mozilla/5.0 (compatible; bingbot/2.0; +http://www.bing.com/bingbot.htm)|12316|
|Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome...|12127|
|Mozilla/5.0 (Linux; Android 6.0.1; Nexus 5X Build/MMB29P) AppleWebKit/537.36 (KHTML, li...|11493|
|                         Mozilla/5.0 (Windows NT 6.1; rv:64.0) Gecko/20100101 Firefox/64.0|10736|
|                  Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/

In [9]:
print('--- HTTP methods ---')
parsed.filter(F.col('_parse_ok')).groupBy('method').count().orderBy(F.desc('count')).show()
spark.stop()
print('Session closed.')


--- HTTP methods ---
+-------+------+
| method| count|
+-------+------+
|    GET|196353|
|   POST|  2718|
|   HEAD|   860|
|OPTIONS|    67|
+-------+------+

Session closed.
